# 03 — Visualization, Cost Analysis & Business Recommendation

**Owner**: Diego  
**Goal**: Visualize model decisions (PCA / t-SNE), perform deep dive on disagreements, run cost analysis with the held-out ground truth, and produce the final business recommendation.

See `TASKS_DIEGO.md` for the detailed checklist.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Make sibling `src/` importable when running the notebook from `notebooks/`.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

from src import evaluation, visualization
print('Modules loaded. ROOT =', ROOT)

### 1.1 Inputs expected from teammates
- `X_processed`, `y_true`, feature names — from **Noah** (`src/preprocessing.py`).
- `outputs/results/preds_*.csv` — from **Isaac** (`src/models.py`).

The cell below loads them defensively: if a file is missing, a clear message tells you what to wait for.

In [ ]:
DATA_PATH = ROOT / 'ai4i2020.csv'
RESULTS_DIR = ROOT / 'outputs' / 'results'
FIGURES_DIR = ROOT / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
y_true = df_raw['Machine failure'].astype(int).to_numpy()
print('Dataset:', df_raw.shape, '| Failure rate:', f'{y_true.mean()*100:.2f}%')

MODEL_KEYS = ['isolation_forest', 'ocsvm', 'lof', 'elliptic']
PRED_FILES = {k: RESULTS_DIR / f'preds_{k}.csv' for k in MODEL_KEYS}

predictions: dict[str, np.ndarray] = {}
for key, path in PRED_FILES.items():
    if path.exists():
        predictions[key] = pd.read_csv(path).iloc[:, 0].to_numpy()
        print(f'OK  {key:18s} {path.name}  ({len(predictions[key])} rows)')
    else:
        print(f'MISSING  {key:18s} expected at {path.relative_to(ROOT)}')

if not predictions:
    print('\n>>> Waiting for Isaac to deliver outputs/results/preds_*.csv. '
          'The rest of this notebook will run as soon as those files appear.')

### 1.2 Processed feature matrix `X_processed`
Noah delivers `build_preprocessor()` (StandardScaler + OneHotEncoder on `Type`). Until that's merged on `main`, we use a local fallback: drop labels and apply the same transformation here so visualisation can run.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

LABEL_COLS = ['UDI', 'Product ID', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
X_df = df_raw.drop(columns=[c for c in LABEL_COLS if c in df_raw.columns])

try:
    from src.preprocessing import build_preprocessor  # type: ignore
    preprocessor = build_preprocessor()
    print('Using Noah\'s build_preprocessor().')
except Exception as exc:  # noqa: BLE001
    print('Noah\'s preprocessor not available, using local fallback. Reason:', exc)
    num_cols = X_df.select_dtypes(include='number').columns.tolist()
    cat_cols = X_df.select_dtypes(exclude='number').columns.tolist()
    preprocessor = ColumnTransformer(
        [('num', StandardScaler(), num_cols),
         ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)],
        remainder='drop',
    )

X_processed = preprocessor.fit_transform(X_df)
if hasattr(X_processed, 'toarray'):
    X_processed = X_processed.toarray()
print('X_processed:', X_processed.shape)

## 2. Visualisations 2D

Goal: project the data into 2D so we can overlay each model's decision and read off (1) where it places its anomaly boundary and (2) where models disagree.

### 2.1 PCA — global variance
Linear, orthogonal, preserves global variance. Cheap and reproducible — our default canvas for overlays.

In [ ]:
X_pca, evr = visualization.pca_2d(X_processed)
print(f'Explained variance ratio: PC1={evr[0]*100:.1f}%, PC2={evr[1]*100:.1f}%, '
      f'cumulative={evr.sum()*100:.1f}%')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.4, s=8)
ax.set_title(f'PCA 2D — {evr.sum()*100:.1f}% variance captured')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.show()

**Takeaway template** — fill in once executed:
> The first 2 PCs capture **{evr_total:.1f}%** of the variance. This suggests a {good/partial} 2D projection. For the deep dive we should keep in mind that the remaining variance lives in the {n_features-2} dropped dimensions.

### 2.2 t-SNE — local neighborhoods
Non-linear, preserves local neighborhoods. Useful to reveal clusters that PCA flattens. We try 3 perplexity values and keep the visually clearest.

In [ ]:
tsne_results = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, p in zip(axes, [5, 30, 50]):
    emb = visualization.tsne_2d(X_processed, perplexity=p)
    tsne_results[p] = emb
    ax.scatter(emb[:, 0], emb[:, 1], alpha=0.4, s=6)
    ax.set_title(f't-SNE (perplexity={p})')
fig.tight_layout()
plt.show()

**Note**: t-SNE distances are **not** preserved globally — read neighborhoods only, not absolute positions. PCA stays our anchor for the model overlay below because a linear projection lets us compare models on the same canvas without re-fitting.

### 2.3 Figure clé — 4 models side by side on PCA
Same canvas, same axes: only the colours change. Differences between subplots are differences between **model decisions**, not between projections.

In [ ]:
if predictions:
    fig = visualization.plot_models_grid(
        X_pca, predictions,
        save_path=str(FIGURES_DIR / 'model_decisions_pca.png'),
    )
    plt.show()
else:
    print('predictions dict is empty — re-run section 1 once Isaac has produced preds_*.csv.')

## 3. Deep dive on disagreement (assessment item 3.3)

> *Identify at least one observation flagged as anomalous by one model but not another. Inspect raw sensor values and explain mathematically or geometrically why this discrepancy occurs based on each model's assumptions.*

Strategy:
1. Build a per-row prediction matrix.
2. Filter rows with disagreement.
3. Pick 3–5 rows that maximise interpretability (different patterns of agreement).
4. Inspect raw sensor values for each.
5. Explain via each algorithm's assumption.

In [ ]:
if not predictions:
    print('Skipping — no predictions yet.')
else:
    pred_df = evaluation.disagreement_matrix(predictions)
    n_disagree = int(pred_df['disagreement'].sum())
    print(f'Disagreement rows: {n_disagree} / {len(pred_df)} '
          f'({n_disagree / len(pred_df) * 100:.1f}%)')
    pred_df.head()

In [ ]:
# Pick the most informative disagreements: one per agreement pattern.
if predictions:
    pattern_col = pred_df[MODEL_KEYS].astype(int).astype(str).agg('|'.join, axis=1)
    cases = (
        pred_df.assign(pattern=pattern_col)
               .loc[pred_df['disagreement']]
               .groupby('pattern')
               .head(1)
               .head(5)
    )
    raw_features = X_df.loc[cases.index]
    deep_dive = pd.concat([raw_features, cases[MODEL_KEYS + ['pattern']]], axis=1)
    deep_dive

### 3.1 Mathematical justification per case

Use the table above and fill one short paragraph per case. Templates (pick the one matching each row):

- **LOF flags, IF does not** — *the row sits in a locally sparse region (few close neighbors → high LOF density ratio), but its axis-aligned coordinates fall in dense intervals so IF isolates it only after many splits — i.e. an inlier in IF's tree depth metric.*
- **Elliptic Envelope flags, LOF does not** — *the row is far from the centre of mass in Mahalanobis distance (long tail of a Gaussian), but it is surrounded by similar extreme neighbours so the local density ratio used by LOF is normal.*
- **IF flags, OC-SVM does not** — *IF separates the row in a few axis-aligned splits (e.g. extreme `Tool wear`). OC-SVM with RBF kernel draws a smooth boundary in feature space; the row's kernel neighbours are dense enough to keep it inside.*
- **OC-SVM flags, IF does not** — *the row sits in a kernel-sparse region (few support-vector neighbours under the chosen γ), but each individual coordinate is in a frequent interval — IF needs deep splits to isolate it.*

## 4. The reveal — comparing to the held-out ground truth

Up to here the notebook has been blind to the `Machine failure` column. We now reintroduce it **only** to score models, never to re-train. Cost convention from the assessment:

- False positive (alert, no failure): **€500** wasted technician time.
- False negative (no alert, failure): **€15 000** unplanned downtime.

FN is **30×** more expensive than FP — the cost-optimal model trades many alerts to catch nearly every failure.

In [ ]:
if not predictions:
    print('Skipping — no predictions yet.')
else:
    comparison = evaluation.compare_models(y_true, predictions)
    display(comparison)

In [ ]:
# Stylised view that's safe to paste in the final report.
if predictions:
    styled = (
        comparison
        .style
        .format({'cost_eur': '€{:,.0f}', 'recall': '{:.2%}', 'precision': '{:.2%}'})
        .background_gradient(subset=['cost_eur'], cmap='Reds')
        .set_caption('Model comparison — sorted by ascending cost (€)')
    )
    styled

**Reading grid**:
- High recall + reasonable precision → low cost (FN dominates).
- Low recall, even with high precision, blows up the cost via FN.
- The best row defines our **baseline**; we tune contamination around it in the next section.